# 02 · Where the delay minutes come from

**RQ1.** How are reported delay minutes distributed across the five BTS causes, and
how stable is that composition across the panel?

The full-panel carrier + late-aircraft group accounts for 73.3% of reported cause minutes
(95% bootstrap interval 73.0-73.6%); NAS + weather + security account for 26.7%.
These descriptive cause groups do not establish airline control or causal responsibility.
Late aircraft may propagate disruption of any origin. The final method audit will
document the bootstrap resampling unit.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

In [ ]:
from team01 import attribution, viz

raw = io.load_flights()
panel = clean.completed(raw)
universe = clean.attribution_universe(raw)

totals = attribution.cause_totals(universe)
totals[["cause", "family", "minutes", "share"]]

## 2.1 The headline split, with an interval

The resampling unit is the **operating day**, not the flight: flights delayed by one
storm are not independent, so a flight-level bootstrap would produce an interval far
too narrow.

In [ ]:
split = attribution.family_split(universe)
interval = attribution.bootstrap_family_share(universe)

print(f"internal share  {interval['share']:.3f}  "
      f"[{interval['ci_low']:.3f}, {interval['ci_high']:.3f}]  "
      f"over {interval['days']} operating days")
print(f"late aircraft   {split['late_aircraft_share']:.1%} of attributed minutes")

## 2.2 Is the composition stable?

Ten years of a small move is the interesting question: if the interval for one year
clears the interval for another, that is a structural break rather than noise.

In [ ]:
attribution.stability_by_year(universe, bootstrap=True, reps=200)

## 2.3 The figure

A unit waffle, because the question is part-to-whole and the cells can be labelled
individually. Area is a weak channel, so the shares are written in the legend rather
than read off the shape.

In [ ]:
viz.fig_cause_waffle(totals)

## Visual audit — Cairo's five questions

### Figure 01 · Where the delay minutes come from (waffle)

| Cairo question | Answer |
|---|---|
| **Guidelines** | One square equals a fixed share of attributed minutes, so area is proportional by construction. Two families only — carrier + late aircraft against NAS + weather + security — and a single accent colour marks the late-aircraft block. No gridlines, no legend box competing with the squares. |
| **Truthful** | The denominator is stated on the figure: attributed minutes on flights arriving 15+ minutes late, 11,454,275 retained rows. The two families sum to 100%, nothing is truncated, and the 73.3% / 26.7% split carries the bootstrap interval [73.0%, 73.6%] in the note rather than in a callout that would imply more precision than the data has. |
| **Functional** | A reader answers "which family dominates, and is one cause bigger than the rest" in one pass: late aircraft alone is 38.8%, larger than all three system causes combined. The two-colour scheme survives greyscale printing. |
| **Beautiful** | A 10×10 grid keeps the module size constant, so a five-square difference reads as five squares rather than as a slightly different rectangle. Title states the finding, not the chart type. |
| **Insightful** | The non-obvious part is the ranking *inside* the internal family: late aircraft outranks carrier delay, which relocates the story from "airlines run late" to "delay is handed on between flights". That is what motivates the rotation analysis in notebook 04. |

### Figure 02 · Cause mix by month

| Cairo question | Answer |
|---|---|
| **Guidelines** | Stacked area on a 0–100% axis, one hue family, direct labels instead of a legend where space allows. Month is ordered, so position encodes order and area encodes share. |
| **Truthful** | The vertical axis is pinned to the full 0–100% range rather than auto-scaled; an auto-scaled axis would turn a two-point seasonal wobble into a dramatic cliff. Shares are of attributed minutes within each month, so months with different volumes stay comparable. |
| **Functional** | The reader can find the late-aircraft band without consulting a legend and can see that it never crosses the 50% line, which keeps the headline consistent with Figure 01. |
| **Beautiful** | Consistent hue order across all months means the eye tracks one band, not five. Thin gaps between bands keep adjacent colours from bleeding. |
| **Insightful** | October and December widen the late-aircraft band while NAS narrows it — the mix shifts toward inherited delay exactly when volume peaks, which is the seasonal signature the airport clusters later pick up. |

### Figure 03 · Attributed delay minutes per delayed flight, by carrier

| Cairo question | Answer |
|---|---|
| **Guidelines** | Horizontal bars sorted by value, one bar per carrier, no colour encoding beyond a single neutral fill: length is the only visual variable, which is the strongest encoding available for a ranking. |
| **Truthful** | The measure is minutes per *delayed* flight among flights with a reconciled cause block, not minutes per flight — stated in the axis label and the note. Carriers with thin cause coverage are not silently dropped; the note names the excluded ones. |
| **Functional** | Sorting answers "who is worst" immediately, and the label column carries the carrier code so no lookup is needed. Values are printed at the bar ends for readers who need exact numbers. |
| **Beautiful** | A single fill avoids the temptation to colour carriers by a third variable that the data does not support. Generous left margin keeps long carrier names legible. |
| **Insightful** | The spread between carriers is far smaller than the spread between causes in Figure 01, so carrier identity explains little of the total — consistent with carrier taking 16.3% of explained variance in the model, against departure hour's 43.7%. |